# 04 - Validate Foundation

Verifies that the initial schemas and Delta tables exist before Bronze
ingestion is implemented or executed.

In [ ]:
import re

dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
catalog = dbutils.widgets.get("catalog").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):
    raise ValueError(f"Invalid catalog identifier: {catalog!r}")

required_tables = [
    "control.processed_files",
    "control.audit_pipeline_runs",
    "control.pipeline_watermarks",
    "control.schema_change_log",
    "control.data_quality_results",
    "control.record_lineage",
    "quarantine.clickstream_errors",
    "quarantine.orders_errors",
    "quarantine.customers_errors",
    "silver.dim_fx_rates",
]

validation_rows = []
for relative_name in required_tables:
    full_name = f"{catalog}.{relative_name}"
    exists = spark.catalog.tableExists(full_name)
    detail = spark.sql(f"DESCRIBE DETAIL {full_name}").first() if exists else None
    validation_rows.append(
        {
            "table_name": full_name,
            "exists": exists,
            "format": detail["format"] if detail else None,
            "num_files": detail["numFiles"] if detail else None,
            "size_in_bytes": detail["sizeInBytes"] if detail else None,
        }
    )

result = spark.createDataFrame(validation_rows)
result.display()

failed = result.filter(
    (~result["exists"]) | (result["format"] != "delta")
).count()
if failed:
    raise RuntimeError(f"Foundation validation failed for {failed} table(s)")

fx_count = spark.table(f"{catalog}.silver.dim_fx_rates").count()
if fx_count == 0:
    raise RuntimeError("silver.dim_fx_rates exists but contains no rows")

print(f"Foundation validation passed. FX rows: {fx_count}")